# Colab 실험 노트북

Google Colab에서 실험을 돌리고 Drive에 결과를 남기기 위한 템플릿입니다.

처음 보는 사람은 위에서부터 실행하면 됩니다. 실제 학습은 기본값으로 꺼두었고, smoke test와 그래프 확인을 먼저 하도록 구성했습니다.

## 1. Drive 연결

Colab 런타임은 꺼지면 파일이 사라질 수 있습니다.

그래서 실험 결과와 백업은 Drive에 저장하는 것이 안전합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 실험 옵션을 어떻게 바꾸나

Colab에서도 실험 방식은 같습니다. 코드를 직접 고치기보다 config와 `RUN_*` 옵션을 바꿉니다.

가장 자주 바꾸는 것은 아래 항목입니다.

| 하고 싶은 일 | 바꿀 곳 | 예시 |
|---|---|---|
| GitHub 저장소 연결 | `REPO_URL` | `https://github.com/team/project.git` |
| Drive 저장 위치 변경 | `DRIVE_ROOT` | `/content/drive/MyDrive/codeit_ml_project` |
| 실제 학습 켜기 | `RUN_REAL_TRAIN` | `True` |
| HuggingFace 확인 켜기 | `RUN_HF_TINY` | `True` |
| 다른 config 실행 | `TEXT_CONFIG`, `RAG_CONFIG`, `REAL_TRAIN_CONFIG` | `configs/my_exp.yaml` |
| 결과 덮어쓰기 방지 | config의 `artifact_policy.run_id` | `run_001` |
| Drive에 결과 저장 | config의 `paths.output_dir`, `backup.backup_dir` | Drive 경로 |

Colab에서 실제 실험을 할 때의 기본 순서는 보통 이렇습니다.

1. `REPO_URL`을 실제 GitHub 주소로 바꿉니다.
2. `DRIVE_ROOT`를 팀 Drive 경로에 맞춥니다.
3. smoke test를 먼저 통과시킵니다.
4. 실제 데이터용 config를 준비합니다.
5. `RUN_REAL_TRAIN = True`로 바꾸고 실행합니다.
6. metric, 학습 곡선, summary를 확인합니다.

주의할 점:

- Colab 런타임은 끊길 수 있으니 결과는 Drive 경로에 저장합니다.
- 실제 학습 전에 항상 smoke test를 먼저 통과시킵니다.
- config의 출력 경로가 로컬 `/content/...`인지 Drive `/content/drive/...`인지 꼭 확인합니다.


## 2. 경로 및 실행 옵션

Colab에서 사용할 저장소 경로와 Drive 경로를 정합니다.

GitHub에 저장소를 올린 뒤 `REPO_URL`을 실제 주소로 바꿔주세요.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

# GitHub 원격 저장소 주소로 바꿔야 합니다.
# GitHub에 올린 뒤 실제 저장소 주소로 바꿉니다.
REPO_URL = "https://github.com/<org>/<repo>.git"

# Colab 런타임 안에 clone할 위치입니다.
# Colab 런타임 안에서 repo를 clone할 위치입니다.
REPO_DIR = Path("/content/codeit-project")

# Drive에 남길 실험 결과/백업/리포트 루트입니다.
# 실험 결과와 백업을 남길 Drive 위치입니다.
DRIVE_ROOT = Path("/content/drive/MyDrive/codeit_ml_project")

# 아래 RUN_* 값으로 실행할 단계를 선택합니다.
RUN_INSTALL = True
RUN_VALIDATE = True
RUN_TEXT_SMOKE = True
RUN_HF_TINY = False
RUN_RAG_SMOKE = True
RUN_RAG_COMPARE = True
# 실제 데이터 학습을 시작할 준비가 되었을 때만 True로 바꿉니다.
RUN_REAL_TRAIN = False
RUN_SUMMARY = True

for path in [DRIVE_ROOT, DRIVE_ROOT / "experiments", DRIVE_ROOT / "backups", DRIVE_ROOT / "reports"]:
    path.mkdir(parents=True, exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DRIVE_ROOT:", DRIVE_ROOT)

## 3. 저장소 준비

Colab 런타임에 저장소를 clone하고 requirements를 설치합니다.

런타임을 새로 시작하면 이 단계부터 다시 실행해야 합니다.

In [ ]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

if RUN_INSTALL:
    !pip install -r requirements.txt

# clone한 repo의 src/ 모듈을 import할 수 있게 경로를 추가합니다.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## 4. 공통 함수

반복해서 사용할 실행/조회/그래프 함수를 정의합니다.

처음 보는 팀원은 함수 내부를 완전히 이해하지 않아도 됩니다. 아래 단계에서 결과를 보기 쉽게 만드는 도구라고 보면 됩니다.

In [ ]:
def run(command: str, check: bool = True) -> subprocess.CompletedProcess:
    # 터미널 명령을 노트북 안에서 실행하고 결과를 바로 보여주는 함수입니다.
    print("\n$", command)
    result = subprocess.run(command, shell=True, text=True, capture_output=True)

    # 표준 출력과 에러를 모두 노트북에 남겨야 나중에 실패 원인을 찾기 쉽습니다.
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # 명령이 실패했는데도 다음 셀이 계속 실행되면 원인이 더 헷갈릴 수 있어서 바로 멈춥니다.
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result


def read_json(path: str | Path) -> dict:
    # metrics.json, run_status.json 같은 결과 파일을 dict로 읽습니다.
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def show_json(path: str | Path) -> dict:
    # JSON 결과를 사람이 읽기 좋게 출력합니다.
    payload = read_json(path)
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    return payload


def show_csv(path: str | Path, n: int = 20) -> pd.DataFrame:
    # CSV 결과를 표 형태로 확인합니다.
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return pd.DataFrame()
    frame = pd.read_csv(path)
    display(frame.head(n))
    return frame


def plot_history(experiment_dir: str | Path) -> pd.DataFrame:
    # history.csv에는 epoch별 metric이 들어갑니다.
    # 이 그래프를 보면 학습이 좋아지는지, 멈춰 있는지, 흔들리는지 볼 수 있습니다.
    history_path = Path(experiment_dir) / "history.csv"
    history = show_csv(history_path)
    if history.empty:
        return history

    metric_cols = [
        col for col in history.columns
        if col != "epoch" and pd.api.types.is_numeric_dtype(history[col])
    ]
    if metric_cols:
        ax = history.plot(x="epoch", y=metric_cols, marker="o", figsize=(8, 4))
        ax.set_title(f"Learning Curve: {Path(experiment_dir).name}")
        ax.set_xlabel("epoch")
        ax.set_ylabel("metric")
        plt.show()
    return history


def plot_metric_bars(metrics: dict, title: str) -> None:
    # metrics.json의 숫자 지표를 막대그래프로 봅니다.
    # 1에 가까울수록 좋은 지표가 많지만, 지표마다 의미는 config와 문서를 함께 확인해야 합니다.
    numeric = {key: value for key, value in metrics.items() if isinstance(value, (int, float))}
    if not numeric:
        print("numeric metric이 없습니다.")
        return

    ax = pd.Series(numeric).sort_index().plot(kind="bar", figsize=(8, 4), ylim=(0, 1.05))
    ax.set_title(title)
    ax.set_ylabel("score")
    plt.xticks(rotation=30, ha="right")
    plt.show()


## 5. GPU 확인

Colab 런타임이 GPU를 잡았는지 확인합니다.

GPU가 없어도 작은 smoke test는 가능하지만, 실제 HuggingFace 학습은 GPU를 권장합니다.

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 6. Config 미리보기

실행 전 config를 먼저 봅니다.

특히 Colab에서는 output 경로와 backup 경로가 Drive를 가리키는지 확인하는 것이 중요합니다.

In [ ]:
import yaml

TEXT_CONFIG = Path("configs/smoke_test_text.yaml")
HF_TINY_CONFIG = Path("configs/smoke_test_hf_tiny.yaml")
RAG_CONFIG = Path("configs/rag_smoke_test.yaml")
REAL_TRAIN_CONFIG = Path("configs/exp002_hf_text_finetune_colab.yaml")
PREDICT_INPUT = Path("data/text_processed/sample_positive.txt")
QUESTION = "예산은 얼마야?"

for config_path in [TEXT_CONFIG, RAG_CONFIG, REAL_TRAIN_CONFIG]:
    print("\n==", config_path, "==")
    if config_path.exists():
        config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
        print(json.dumps({
            "experiment": config.get("experiment"),
            "paths": config.get("paths"),
            "model": config.get("model"),
            "rag_retriever": config.get("rag", {}).get("retriever"),
            "backup": config.get("backup"),
        }, ensure_ascii=False, indent=2))

## 7. Smoke Test 실행

Colab 환경에서 기본 파이프라인이 도는지 확인합니다.

실제 데이터를 넣기 전에 이 단계가 먼저 통과해야 합니다.

In [ ]:
if RUN_VALIDATE:
    run("python scripts/run_validate.py --data-dir data/text_processed --project-root .")

if RUN_TEXT_SMOKE:
    run(f"python scripts/run_train.py --config {TEXT_CONFIG} --project-root .")
    run(f"python scripts/run_predict.py --config {TEXT_CONFIG} --project-root . --input {PREDICT_INPUT}")

### Smoke Test Metric과 학습 곡선

Colab에서 만든 smoke test 결과를 확인합니다.

로컬과 비슷한 결과가 나오면 환경 차이로 인한 문제는 크지 않다고 볼 수 있습니다.

In [ ]:
text_dir = Path("experiments/smoke_test_text")
text_metrics = show_json(text_dir / "metrics.json")
plot_metric_bars(text_metrics, "Text Classification Metrics")
plot_history(text_dir)

## 8. HuggingFace Tiny 확인

HuggingFace 모델 실행 환경을 확인하는 선택 단계입니다.

시간이 더 걸릴 수 있으므로 기본값은 꺼두었습니다. 실제 fine-tuning 전에 한 번 켜보면 좋습니다.

In [ ]:
if RUN_HF_TINY:
    run(f"python scripts/run_train.py --config {HF_TINY_CONFIG} --project-root .")
    run(f"python scripts/run_predict.py --config {HF_TINY_CONFIG} --project-root . --input {PREDICT_INPUT}")

    hf_dir = Path("experiments/smoke_test_hf_tiny")
    plot_metric_bars(show_json(hf_dir / "metrics.json"), "HF Tiny Metrics")
    plot_history(hf_dir)
else:
    print("skip")

## 9. RAG Smoke Test와 평가 그래프

Colab에서도 RAG 파이프라인이 끝까지 도는지 확인합니다.

검색, 답변, citation 지표를 그래프로 보고 이상한 점이 있는지 확인합니다.

In [ ]:
if RUN_RAG_SMOKE:
    run(f"python scripts/run_rag_ingest.py --config {RAG_CONFIG} --project-root .")
    run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --question '{QUESTION}'")
    run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --evaluate")

rag_dir = Path("experiments/rag_smoke_test")
rag_metrics = show_json(rag_dir / "metrics.json")
plot_metric_bars(rag_metrics, "RAG Evaluation Metrics")
show_csv(rag_dir / "evaluation_results.csv")

## 10. Retriever 비교

검색 방식을 비교합니다.

실제 RFP 문서가 들어오면 keyword, semantic, hybrid 중 어떤 방식이 더 나은지 이 표와 그래프로 판단합니다.

In [ ]:
if RUN_RAG_COMPARE:
    run("python scripts/compare_rag_retrievers.py --project-root .")

comparison = show_csv("reports/rag_retriever_comparison.csv")
if not comparison.empty:
    metric_cols = ["retrieval_hit_rate", "answer_contains_expected_rate", "citation_correct_rate", "not_found_rate"]
    existing = [col for col in metric_cols if col in comparison.columns]
    comparison.set_index("experiment")[existing].plot(kind="bar", figsize=(9, 4), ylim=(0, 1.05))
    plt.title("RAG Retriever Comparison")
    plt.xticks(rotation=30, ha="right")
    plt.show()

## 11. 실제 데이터 학습

실제 프로젝트 데이터가 Drive에 준비된 뒤 실행하는 단계입니다.

기본값은 꺼두었습니다. config의 데이터 경로와 출력 경로를 확인한 뒤 `RUN_REAL_TRAIN = True`로 바꿔 실행하세요.

In [ ]:
if RUN_REAL_TRAIN:
    run(f"python scripts/run_train.py --project-root . --config {REAL_TRAIN_CONFIG}")

    real_config = yaml.safe_load(REAL_TRAIN_CONFIG.read_text(encoding="utf-8"))
    real_output = Path(real_config.get("paths", {}).get("output_dir", ""))
    plot_metric_bars(show_json(real_output / "metrics.json"), "Real Train Metrics")
    plot_history(real_output)
else:
    print("skip")

## 12. 실험 요약

여러 실험 결과를 표로 모읍니다.

팀 공유나 발표 자료를 만들 때 이 표를 기준으로 어떤 실험이 더 나았는지 비교합니다.

In [ ]:
if RUN_SUMMARY:
    run("python scripts/summarize_experiments.py --project-root .")

summary = show_csv("reports/experiment_summary.csv")

## 13. 실험 메모

Colab 런타임은 꺼질 수 있으니, 실험 후 판단을 꼭 남겨둡니다.

- 이번 실험 목적:
- metric에서 볼 점:
- 학습 곡선에서 이상한 점:
- RAG 검색/답변에서 보완할 점:
- 다음 실험에서 바꿀 config: